In [2]:
import numpy as np
from pathlib import Path
import sys

for path in (Path.cwd(), Path.cwd() / "certificates" / "empirical_laws", Path.cwd().parent / "certificates" / "empirical_laws"):
    if (path / "notebook_setup.py").exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

import notebook_setup
from lyapunov import has_lyapunov
from utils import dask_parallel_map


# User-configurable parameters
EPSILONS = np.linspace(0.05, 0.95, 50)
L_VALUES = [1.0, 10, 100, 1000]
KAPPA_VALUES = [2, 10, 100, 1000, 10000]
N_WORKERS = 2  # Empirical Law 4.3 cubic rate is only defined for n=2.

TEST_IMPROVEMENT = 1 - 1e-2
MOSEK_SOLVE_KWARGS = notebook_setup.MOSEK_STRICT_SOLVE_KWARGS
SOLVER = "MOSEK"
METHODS = ("EF21", "EF")
DASK_SCHEDULER = "processes"
DASK_NUM_WORKERS = 14
def has_rate_clean(rho, eta, mus, Ls, eps, *, method):
    return notebook_setup.warning_aware_solve(
        has_lyapunov,
        float(rho),
        eta=float(eta),
        delta=1 - eps,
        n_workers=N_WORKERS,
        mus=mus,
        Ls=Ls,
        method=method,
        use_simplified_lyapunov=False,
        homogenous=method == "EF",
        solver=SOLVER,
        solve_kwargs=MOSEK_SOLVE_KWARGS,
    )


def verify_config(args):
    L_tuple, kappa_tuple, eps = args
    scale_info = notebook_setup.scaled_problem_data_for_case(L_tuple, kappa_tuple)
    Ls = scale_info["Ls"]
    mus = scale_info["mus"]
    scale_label = notebook_setup.scale_label(scale_info)

    rho = notebook_setup.rho_opt_n2(eps, Ls, mus)
    if not np.isfinite(rho):
        return (
            f"FAIL: L={L_tuple} kappa={kappa_tuple} Eps={eps:.3f} | "
            f"{scale_label}: no finite valid cubic root"
        )

    eta = notebook_setup.empirical_eta_star(eps, Ls, mus, N_WORKERS)
    reference_sources = []
    for method in METHODS:
        ref_info = has_rate_clean(rho, eta, mus, Ls, eps, method=method)
        reference_source = f"{method}:{SOLVER}"
        if not ref_info["ok_clean"]:
            return (
                f"FAIL: L={L_tuple} kappa={kappa_tuple} Eps={eps:.3f} | "
                f"{scale_label}: {method} reference not clean "
                f"(source={reference_source} raw_ok={ref_info['ok_raw']} "
                f"warnings={ref_info['warn_patterns']})"
            )
        reference_sources.append(reference_source)
    reference_source = ",".join(reference_sources)

    rho_imp = rho * TEST_IMPROVEMENT
    for method in METHODS:
        imp_info = has_rate_clean(rho_imp, eta, mus, Ls, eps, method=method)
        if imp_info["ok_clean"]:
            return (
                f"FAIL: L={L_tuple} kappa={kappa_tuple} Eps={eps:.3f} | "
                f"{scale_label}: {method} cleanly certifies an improved rate "
                f"rho_opt={rho:.8f} rho_imp={rho_imp:.8f} reference_source={reference_source}"
            )
    return None


def run_rigorous_check():
    worker_configs = notebook_setup.worker_L_kappa_configs(
        L_VALUES, KAPPA_VALUES, N_WORKERS, dedup_permutations=True
    )
    configs = [
        (L_cfg, kappa_cfg, float(eps))
        for (L_cfg, kappa_cfg) in worker_configs
        for eps in EPSILONS
    ]
    print(f"--- Verification Suite: Empirical Law 4.3 (n=2 Rate), n={N_WORKERS} ---")
    print(f"Checking {len(configs)} configurations with {DASK_NUM_WORKERS} workers...")
    results = dask_parallel_map(
        verify_config,
        configs,
        scheduler=DASK_SCHEDULER,
        num_workers=DASK_NUM_WORKERS,
    )
    failures = [msg for msg in results if msg]

    if not failures:
        print("ALL CHECKS PASSED\n")
        return []

    for msg in failures:
        print(msg)
    print(f"FAILURES={len(failures)}\n")
    return failures


if __name__ == "__main__":
    total_failures = run_rigorous_check()
    if total_failures:
        print(f"TOTAL_FAILURES={len(total_failures)}")
        raise SystemExit(1)
    print("ALL N=2 RATE CHECKS PASSED")

--- Verification Suite: Empirical Law 4.3 (n=2 Rate), n=2 ---
Checking 10500 configurations with 14 workers...


compute: 100%|██████████| 10500/10500 [01:10<00:00, 148.86it/s]


ALL CHECKS PASSED

ALL N=2 RATE CHECKS PASSED
